This script only runs from within Meteowiss, because of the data store. It compiles raw data files stored on MeteoSwiss disk into parquet files. 
Incoming raw data files are first organized into folders, and a stastic is computed and displayed showing the number of recently incoming files.
Then, yearly files are generated for Meteo bulletins, Thermo zip files, NOAA CPD2 tarballs. Monthly files are generated for AE33 zip files, G2401 tarballs.

joerg.klausen@meteoswiss.ch

[TODO] Improve handling of erroneous files from g2401.compile_g2401_to_parquet

In [ ]:
import os

import pandas as pd
import polars as pl

import housekeeping.organize_files as hk
import monitoring.file_coverage as fc
from processing.ae33 import AE33
from processing.cpd2 import CPD2
from processing.dobson import DData
from processing.g2401 import G2401
from processing.meteo import Meteo
from processing.ne300 import NE300
from processing.thermo import Thermo
from toolbox.utils import load_config, setup_logging

##############################
# Process MKN incoming files #
##############################
# read configuration
mkn = load_config(config_file="mch-mkn.yml")

# setup logging
logger = setup_logging(os.path.join(os.getcwd(), mkn['logging']))

# organize MKN files on MeteoSwiss fileshare
n = hk.organize_files(mkn, branch=mkn['branches']["incoming"], verbosity=1)

for k, v in mkn['branches'].items():
    if not os.path.exists(os.path.join(mkn['root'], v)):
        raise ValueError(f"path '{v}' not found.")

# path of source files
incoming = os.path.join(mkn['root'], mkn['branches']['incoming'])

# path of archive for successfully processed raw data files.
archive = os.path.join(mkn['root'], mkn['branches']['archive'])

# path for raw data files with issues
issues = os.path.join(mkn['root'], mkn['branches']['issues'])

# path for log files
logs = os.path.join(mkn['root'], mkn['branches']['logs'])

# relative path of level1 data .parquet files on repo
level1 = os.path.join("data", "level1", "mkn")
os.makedirs(level1, exist_ok=True)

# process all raw data types
years = [f"{year}" for year in range(2023, 2026)]
months = ["{:02d}".format(mm) for mm in range(1, 13, 1)]

In [ ]:
# show and plot file statistics
days = 7
stats = fc.get_file_coverage(cfg=mkn, days=days)
fc.plot_file_coverage(stats=stats)

# stats = hk.get_file_counts(base_path=mkn['root'], base_name='tei49c', base_folders=mkn['branches'])
# hk.plot_file_counts(df=stats)

In [ ]:
met = Meteo(log=os.path.join(logs, "meteo.log"))
for year in years:
    met.compile_vrxa00_to_parquet(
        source=os.path.join(incoming, "meteo", year), 
        target=os.path.join(level1, year), 
        archive=os.path.join(archive, "meteo", year), 
        issues=os.path.join(issues, "meteo"),
        )

In [ ]:
# thermo = Thermo(log=os.path.join(logs, "thermo.log"))
thermo = Thermo(config=mkn)
for year in years:
    thermo.compile_thermo_to_parquet(
        source=os.path.join(incoming, "tei49c", year),
        target=os.path.join(level1, year),
        archive=os.path.join(archive, "tei49c", year),
        issues=os.path.join(issues, "tei49c"),
        )        

In [ ]:
thermo = Thermo(config=mkn)
for year in years:
    thermo.compile_thermo_to_parquet(
        source=os.path.join(incoming, "tei49i", year),
        target=os.path.join(level1, year),
        archive=os.path.join(archive, "tei49i", year),
        issues=os.path.join(issues, "tei49i"),
        )

In [ ]:
# from toolbox.utils import pl_simplify_dtypes
# thermo = Thermo(config=mkn)
# file = 'data/level1/mkn/2025/tei49i.parquet'
# df = pl.read_parquet(file)
# df = pl_simplify_dtypes(df, digits=1)
# df = df.with_columns(pl.col("dtm").cast(pl.Datetime("us")).dt.replace_time_zone("UTC"))
# df_col_names = df.columns
# print(df.schema)

# df2 = thermo.extract_lrec_from_file("tei49i_all_lrec-20250310204725.dat")
# df2 = df2.unique().sort('dtm')
# df2 = df2.with_columns(pl.col("dtm").cast(pl.Datetime("us")).dt.replace_time_zone("UTC"))
# print(df2.schema)

# df = df.select(df_col_names)
# df2 = df2.select(df_col_names)


# df3 = pl.concat([df, df2]).unique(subset=["dtm"], keep="first").sort("dtm")  # Optional: Sort by datetime if needed
# print(df3.schema)
# df3.write_parquet(f"{file}_")

In [ ]:
ae33 = AE33(log=os.path.join(logs, "ae33.log"))
for year in years:
    for month in months:
        ae33.zipfiles_to_parquet(
            source=os.path.join(incoming, "ae33", "data", year, month), 
            target=os.path.join(level1, year, month), 
            archive=os.path.join(archive, "ae33", "data", year, month), 
            issues=os.path.join(issues, "ae33"), 
            plot=True,
            )

In [ ]:
# for data fetched directly from logger
# ae33 = AE33(log=os.path.join(logs, "ae33.log"))
# year = '2025'
# month = '99'
# ae33.zipfiles_to_parquet(
#     source=os.path.join(incoming, "ae33", "data", month), 
#     target=os.path.join(level1, year, month), 
#     archive=os.path.join(archive, "ae33", "data", year, month), 
#     issues=os.path.join(issues, "ae33"), 
#     plot=True,
#     )

In [ ]:
g2401 = G2401(log=os.path.join(logs, "g2401.log"))
for year in years:
    for month in months:
        g2401.compile_g2401_to_parquet(
            source=os.path.join(incoming, "g2401", year, month), 
            target=os.path.join(level1, year, month), 
            archive=os.path.join(archive, "g2401", year, month), 
            issues=os.path.join(issues, "g2401", year),
            )

In [ ]:
ne300 = NE300(config=mkn)

# [TODO] fix!!
ne300.compile_files_to_parquet(
    source=os.path.join(incoming, "ne300"),#, year, month), 
    target=level1,#os.path.join(level1, year, month), 
    move_processed_files=True,
    split='1mo', 
    )

In [ ]:
# # read configuration
# mkn = load_config(config_file="mch-mkn.yml")

# ne300 = NE300(config=mkn)

# ne300.compile_files_to_parquet()

In [ ]:
# display empty directories under source
# os.system(f"find {source} -empty -type d)

# display empty directories under source (NB: no questions asked!)
# os.system(f"find {source} -empty -type d -delete")

In [ ]:
##############################
# Process NRB incoming files #
##############################
# read configuration
nrb = load_config(config_file="mch-nrb.yml")

# setup logging
logger = setup_logging(os.path.join(os.getcwd(), nrb["nrb-dobson"]['logging']))


n = hk.organize_files(nrb["nrb-dobson"], branch="uploads")
print(f"Finished organizing files under '{nrb['nrb-dobson']['root']}'. {n} files moved.")

days = 60
stats = fc.get_file_coverage(cfg=nrb['nrb-dobson'], days=days)
# fc.plot_coverage(stats=stats, days=days)
# fc.print_coverage(stats=stats, days=days)

ddata = DData(config=nrb['nrb-dobson'])
headers, df_data = ddata.process_directory(os.path.join(nrb['nrb-dobson']["root"], nrb['nrb-dobson']['branches']['uploads']))
df_data.write_parquet(os.path.join(level1, 'd018_nrb_data.parquet'))

df_headers = pl.DataFrame(headers, orient='row')
df_headers.write_parquet(os.path.join(level1, 'd018_nrb_headers.parquet'))

In [ ]:
file = 'data/level1/nrb/d018_nrb_data.parquet'
df = pl.read_parquet(file)
df = df.with_columns(pl.col('dtm').str.to_datetime(),
                     pl.col('sza').cast(pl.Float32),
                     pl.col('mu').cast(pl.Float32),
                     pl.col('XAD').cast(pl.Float32),
                     pl.col('X1').cast(pl.Float32),
                     pl.col('X2').cast(pl.Float32),
                     pl.col('X3').cast(pl.Float32),
                     pl.col('X4').cast(pl.Float32),
                     pl.col('X5').cast(pl.Float32),)
display(df.describe())

display(df.plot(x='dtm', y='sza'))
display(df.plot(x='dtm', y='mu'))
display(df.plot(x='dtm', y='XAD'))

In [ ]:
# Brewer B071 file statistics
days = 7
stats = fc.get_file_coverage(cfg=nrb["nrb-brewer"], days=days)
fc.plot_coverage(stats=stats, days=days)
fc.print_coverage(stats=stats, days=days)